# Phase 1 · Taxonomy and host identities

Taxonomy records are versioned inputs to all later stages. V4 keeps taxonomy acquisition separate from cohort selection: the same normalized records can support human, livestock, crop, or mixed runs.

This notebook checks the pinned NCBI host taxonomy, inspects the converted pathogen taxonomy, reports group-level counts, and checks alias/variant references.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository')
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, log_step
ROOT, LOG, LOG_PATH = configure_notebook('01_taxonomy_hosts', 'pilot-v1')
print('Project root:', ROOT, '| log:', LOG_PATH)

2026-11-02 11:25:10,792 INFO Notebook session started at 2026-11-02T11:25:10.792807+00:00


2026-11-02 11:25:10,793 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4


2026-11-02 11:25:10,793 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/01_taxonomy_hosts.log


Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4 | log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/01_taxonomy_hosts.log


## Step 1 · Inspect the pinned host taxonomy release

In [2]:
import json
from IPython.display import display
manifest_path = ROOT / 'data/source_taxonomy/ncbi/source_manifest.json'
manifest = json.loads(manifest_path.read_text())
print('Manifest:', manifest_path)
display({k: manifest.get(k) for k in ('release', 'archive', 'sha256', 'retrieved_at') if k in manifest})
LOG.info('Loaded host taxonomy manifest: %s', manifest_path)

Manifest: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/source_taxonomy/ncbi/source_manifest.json


{}

2026-11-02 11:25:10,799 INFO Loaded host taxonomy manifest: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/source_taxonomy/ncbi/source_manifest.json


## Step 2 · Load the normalized V4 taxonomy tables

In [3]:
import pandas as pd
STAGING_ROOT = sorted((ROOT / 'data/v4/staging').glob('*/taxonomy'))[-1].parent
TAXONOMY_ROOT = STAGING_ROOT / 'taxonomy'
hosts = pd.read_parquet(TAXONOMY_ROOT / 'hosts.parquet')
pathogens = pd.read_parquet(TAXONOMY_ROOT / 'pathogens.parquet')
aliases = pd.read_parquet(TAXONOMY_ROOT / 'pathogen_aliases.parquet')
variants = pd.read_parquet(TAXONOMY_ROOT / 'pathogen_variants.parquet')
print('Staging release:', STAGING_ROOT)
print('Loaded hosts:', len(hosts), '| pathogens:', len(pathogens), '| aliases:', len(aliases), '| variants:', len(variants))
LOG.info('Loaded taxonomy tables hosts=%s pathogens=%s aliases=%s variants=%s', len(hosts), len(pathogens), len(aliases), len(variants))

2026-11-02 11:25:11,450 INFO Loaded taxonomy tables hosts=5 pathogens=142201 aliases=233550 variants=19285


Staging release: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/staging/v3-to-v4-20261102-ncbi-f278697a8ba6-v3
Loaded hosts: 5 | pathogens: 142201 | aliases: 233550 | variants: 19285


## Step 3 · Summarize pathogen groups and taxonomy sources

In [4]:
display(pathogens.groupby(['pathogen_group', 'taxonomy_source'], dropna=False).size().rename('records').to_frame())
display(hosts[['host_id', 'accepted_name', 'host_group', 'taxonomy_release']].sort_values(['host_group', 'accepted_name']))
LOG.info('Pathogen group totals: %s', pathogens.groupby('pathogen_group').size().to_dict())

,,records
pathogen_group,taxonomy_source,
bacteria,lpsn,22804
fungi,species_fungorum,101843
viruses,ictv_msl,17554


,host_id,accepted_name,host_group,taxonomy_release
0,ncbi:4530,Oryza sativa,crop,ncbi-taxdump-2026-11-02-sha256-f278697a8ba6
1,ncbi:4577,Zea mays,crop,ncbi-taxdump-2026-11-02-sha256-f278697a8ba6
2,ncbi:9606,Homo sapiens,human,ncbi-taxdump-2026-11-02-sha256-f278697a8ba6
3,ncbi:9913,Bos taurus,livestock,ncbi-taxdump-2026-11-02-sha256-f278697a8ba6
4,ncbi:9940,Ovis aries,livestock,ncbi-taxdump-2026-11-02-sha256-f278697a8ba6


2026-11-02 11:25:11,468 INFO Pathogen group totals: {'bacteria': 22804, 'fungi': 101843, 'viruses': 17554}


## Step 4 · Check normalized references

In [5]:
pathogen_ids = set(pathogens.pathogen_id.astype(str))
linked_aliases = aliases.loc[aliases.pathogen_id.notna()] if len(aliases) else aliases
unknown_aliases = int((~linked_aliases.pathogen_id.astype(str).isin(pathogen_ids)).sum()) if len(linked_aliases) else 0
unresolved_aliases = int(aliases.pathogen_id.isna().sum()) if len(aliases) else 0
unknown_variants = int((~variants.pathogen_id.astype(str).isin(pathogen_ids)).sum()) if len(variants) else 0
TAXONOMY_CHECKS = {'unique_pathogen_ids': pathogens.pathogen_id.is_unique,
                   'unique_host_ids': hosts.host_id.is_unique,
                   'aliases_without_pathogen': unknown_aliases,
                   'unresolved_aliases_accounted': unresolved_aliases,
                   'variants_without_pathogen': unknown_variants}
display(TAXONOMY_CHECKS)
if unknown_aliases or unknown_variants or not pathogens.pathogen_id.is_unique or not hosts.host_id.is_unique:
    raise ValueError('Taxonomy reference checks failed; inspect the normalized tables before acquisition')
LOG.info('Taxonomy checks passed; unresolved aliases retained as unmapped rows: %d', unresolved_aliases)
print('Unresolved alias rows are retained with explicit missing pathogen IDs:', unresolved_aliases)

{'unique_pathogen_ids': True,
 'unique_host_ids': True,
 'aliases_without_pathogen': 0,
 'unresolved_aliases_accounted': 21291,
 'variants_without_pathogen': 0}

2026-11-02 11:25:11,548 INFO Taxonomy checks passed; unresolved aliases retained as unmapped rows: 21291


Unresolved alias rows are retained with explicit missing pathogen IDs: 21291


## Step 5 · Host identifiers available to run configuration

In [6]:
HOST_LOOKUP = hosts[['host_id', 'accepted_name', 'host_group', 'taxon_id']].sort_values('host_id')
display(HOST_LOOKUP)
print('Use host_id values in run configuration; names and aliases are search terms.')
LOG.info('Host selectors available: %s', HOST_LOOKUP.host_id.tolist())

,host_id,accepted_name,host_group,taxon_id
0,ncbi:4530,Oryza sativa,crop,4530
1,ncbi:4577,Zea mays,crop,4577
2,ncbi:9606,Homo sapiens,human,9606
3,ncbi:9913,Bos taurus,livestock,9913
4,ncbi:9940,Ovis aries,livestock,9940


2026-11-02 11:25:11,554 INFO Host selectors available: ['ncbi:4530', 'ncbi:4577', 'ncbi:9606', 'ncbi:9913', 'ncbi:9940']


Use host_id values in run configuration; names and aliases are search terms.
